# Lab 06 — Support Vector Machine (SVM)
**Học máy ứng dụng · Nguyễn Hoàng Hải · 2474802010099**

Notebook bám theo tài liệu Lab06_SVM.pdf. Chạy lần lượt từ trên xuống. Bộ dữ liệu CSV được tìm ở thư mục hiện tại hoặc `data/`.

## 3. Lề mềm, tham số C và chuẩn hoá

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def doc_csv(ten):
    ung_vien = [Path('data') / ten, Path(ten)]
    if ten == 'sinh_vien.csv':
        ung_vien += [Path('sinh_vien (1).csv'), Path('data/sinh_vien (1).csv')]
    for p in ung_vien:
        if p.exists():
            print(f'Đọc dữ liệu: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(f'Không tìm thấy {ten}. Đã tìm: {ung_vien}')

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df=doc_csv('sinh_vien.csv')
X=df[['gio_on','diem_giua_ky']]; y=df['qua_mon']
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
rows=[]
for C in [.01,.1,1,10,100]:
    model=make_pipeline(StandardScaler(),SVC(kernel='linear',C=C)).fit(X_tr,y_tr)
    svc=model[-1]
    rows.append({'C':C,'support_vectors':len(svc.support_),'margin_width_scaled':2/np.linalg.norm(svc.coef_[0]),'train_accuracy':accuracy_score(y_tr,model.predict(X_tr)),'test_accuracy':accuracy_score(y_te,model.predict(X_te))})
pd.DataFrame(rows).round(4)

In [ ]:
df3=doc_csv('sinh_vien_mo_rong.csv')
X3=df3[['gio_on','diem_giua_ky','chi_tieu']]; y3=df3['qua_mon']
X3_tr,X3_te,y3_tr,y3_te=train_test_split(X3,y3,test_size=.25,random_state=17,stratify=y3)
raw=SVC().fit(X3_tr,y3_tr)
scaled=make_pipeline(StandardScaler(),SVC()).fit(X3_tr,y3_tr)
print(f'Tỷ lệ lớp qua ở tập kiểm tra: {y3_te.mean():.4f}')
print(f'Không chuẩn hoá: {accuracy_score(y3_te,raw.predict(X3_te)):.4f}')
print(f'Có chuẩn hoá: {accuracy_score(y3_te,scaled.predict(X3_te)):.4f}')

**Nhận xét:** C nhỏ phạt vi phạm lề nhẹ, thường tạo lề rộng và nhiều support vectors. Khi các đặc trưng có đơn vị khác nhau (đặc biệt `chi_tieu`), chuẩn hoá trong pipeline ngăn cột số lớn lấn át các cột còn lại.